<center><img src="car.jpg" width=500></center>


Insurance companies invest a lot of time and money into optimizing their pricing and accurately estimating the likelihood that customers will make a claim. In many countries insurance it is a legal requirement to have car insurance in order to drive a vehicle on public roads, so the market is very large!

(`Source: https://www.accenture.com/_acnmedia/pdf-84/accenture-machine-leaning-insurance.pdf`) 

Knowing all of this, On the Road car insurance have requested your services in **building a model to predict whether a customer will make a claim on their insurance during the policy period**. As they have very little expertise and infrastructure for deploying and monitoring machine learning models, they've asked you to **identify the single feature that results in the best performing model, as measured by accuracy**, so they can start with a simple model in production.

They have supplied you with their customer data as a csv file called `car_insurance.csv`, along with a table detailing the column names and descriptions below.



## The dataset

| Column | Description |
|--------|-------------|
| `id` | Unique client identifier |
| `age` | Client's age: <br> <ul><li>`0`: 16-25</li><li>`1`: 26-39</li><li>`2`: 40-64</li><li>`3`: 65+</li></ul> |
| `gender` | Client's gender: <br> <ul><li>`0`: Female</li><li>`1`: Male</li></ul> |
| `driving_experience` | Years the client has been driving: <br> <ul><li>`0`: 0-9</li><li>`1`: 10-19</li><li>`2`: 20-29</li><li>`3`: 30+</li></ul> |
| `education` | Client's level of education: <br> <ul><li>`0`: No education</li><li>`1`: High school</li><li>`2`: University</li></ul> |
| `income` | Client's income level: <br> <ul><li>`0`: Poverty</li><li>`1`: Working class</li><li>`2`: Middle class</li><li>`3`: Upper class</li></ul> |
| `credit_score` | Client's credit score (between zero and one) |
| `vehicle_ownership` | Client's vehicle ownership status: <br><ul><li>`0`: Does not own their vehilce (paying off finance)</li><li>`1`: Owns their vehicle</li></ul> |
| `vehcile_year` | Year of vehicle registration: <br><ul><li>`0`: Before 2015</li><li>`1`: 2015 or later</li></ul> |
| `married` | Client's marital status: <br><ul><li>`0`: Not married</li><li>`1`: Married</li></ul> |
| `children` | Client's number of children |
| `postal_code` | Client's postal code | 
| `annual_mileage` | Number of miles driven by the client each year |
| `vehicle_type` | Type of car: <br> <ul><li>`0`: Sedan</li><li>`1`: Sports car</li></ul> |
| `speeding_violations` | Total number of speeding violations received by the client | 
| `duis` | Number of times the client has been caught driving under the influence of alcohol |
| `past_accidents` | Total number of previous accidents the client has been involved in |
| `outcome` | Whether the client made a claim on their car insurance (response variable): <br><ul><li>`0`: No claim</li><li>`1`: Made a claim</li></ul> |

In [2]:
# import modules
import pandas as pd
import numpy as np
from statsmodels.formula.api import logit

# load the dataset
car = pd.read_csv('car_insurance.csv')

# EDA
display(car.head())
car.info()
display(car.describe().round(2))

,id,age,gender,driving_experience,education,income,credit_score,vehicle_ownership,vehicle_year,married,children,postal_code,annual_mileage,vehicle_type,speeding_violations,duis,past_accidents,outcome
0,569520,3,0,0-9y,high school,upper class,0.629027,1.0,after 2015,0.0,1.0,10238,12000.0,sedan,0,0,0,0.0
1,750365,0,1,0-9y,none,poverty,0.357757,0.0,before 2015,0.0,0.0,10238,16000.0,sedan,0,0,0,1.0
2,199901,0,0,0-9y,high school,working class,0.493146,1.0,before 2015,0.0,0.0,10238,11000.0,sedan,0,0,0,0.0
3,478866,0,1,0-9y,university,working class,0.206013,1.0,before 2015,0.0,1.0,32765,11000.0,sedan,0,0,0,0.0
4,731664,1,1,10-19y,none,working class,0.388366,1.0,before 2015,0.0,0.0,32765,12000.0,sedan,2,0,1,1.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   10000 non-null  int64  
 1   age                  10000 non-null  int64  
 2   gender               10000 non-null  int64  
 3   driving_experience   10000 non-null  object 
 4   education            10000 non-null  object 
 5   income               10000 non-null  object 
 6   credit_score         9018 non-null   float64
 7   vehicle_ownership    10000 non-null  float64
 8   vehicle_year         10000 non-null  object 
 9   married              10000 non-null  float64
 10  children             10000 non-null  float64
 11  postal_code          10000 non-null  int64  
 12  annual_mileage       9043 non-null   float64
 13  vehicle_type         10000 non-null  object 
 14  speeding_violations  10000 non-null  int64  
 15  duis                 10000 non-null  

,id,age,gender,credit_score,vehicle_ownership,married,children,postal_code,annual_mileage,speeding_violations,duis,past_accidents,outcome
count,10000.00,10000.00,10000.0,9018.00,10000.00,10000.0,10000.00,10000.00,9043.00,10000.00,10000.00,10000.00,10000.00
mean,500521.91,1.49,0.5,0.52,0.70,0.5,0.69,19864.55,11697.00,1.48,0.24,1.06,0.31
std,290030.77,1.03,0.5,0.14,0.46,0.5,0.46,18915.61,2818.43,2.24,0.55,1.65,0.46
min,101.00,0.00,0.0,0.05,0.00,0.0,0.00,10238.00,2000.00,0.00,0.00,0.00,0.00
25%,249638.50,1.00,0.0,0.42,0.00,0.0,0.00,10238.00,10000.00,0.00,0.00,0.00,0.00
50%,501777.00,1.00,0.0,0.53,1.00,0.0,1.00,10238.00,12000.00,0.00,0.00,0.00,0.00
75%,753974.50,2.00,1.0,0.62,1.00,1.0,1.00,32765.00,14000.00,2.00,0.00,2.00,1.00
max,999976.00,3.00,1.0,0.96,1.00,1.0,1.00,92101.00,22000.00,22.00,6.00,15.00,1.00


In [3]:
# imput missing data to the mean
car['credit_score'].fillna(car['credit_score'].mean(),inplace=True)
car['annual_mileage'].fillna(car['annual_mileage'].mean(),inplace=True)

# check missing values again
print(f"The DataFrame has {car.isna().any().sum()} missing values.")

The DataFrame has 0 missing values.


* Identify the single feature of the data that is the **best predictor** of whether a customer will put in a claim ("outcome" column), excluding the "id" column.

In [4]:
# create a variable for the features (all columns except "outcome" and "id")
features_cols = car.columns.drop(["outcome","id"])

# create and empty list to store each model object from the for loop
models = []
# loop through features and create a Logistic Regression model to estimate the relationship between "outcome" and the iterator from features, fit the data
for col in features_cols:
    model = logit(f"outcome~{col}",data=car).fit(disp=0)
    models.append(model)

* Store as a DataFrame called best_feature_df, containing columns named **"best_feature"** and **"best_accuracy"** with the name of the feature with the highest accuracy, and the respective accuracy score.

In [5]:
## Measuring performance

# create and empty list to store accuracies
accuracy = []
# interate over the index of models, create a confusion matrix and store individual metrics
for col in range(len(models)):
    conf_matrix = models[col].pred_table()
    tn = conf_matrix[0,0]
    tp = conf_matrix[1,1]
    fn = conf_matrix[1,0]
    fp = conf_matrix[0,1]
    acc = (tn+tp)/(tn+tp+fn+fp)
    accuracy.append(acc)
# print each column's accuracy
for col, acc in zip(features_cols, accuracy):
    print(f"The column '{col}' has accuracy: {acc}")

The column 'age' has accuracy: 0.7747
The column 'gender' has accuracy: 0.6867
The column 'driving_experience' has accuracy: 0.7771
The column 'education' has accuracy: 0.6867
The column 'income' has accuracy: 0.7425
The column 'credit_score' has accuracy: 0.7054
The column 'vehicle_ownership' has accuracy: 0.7351
The column 'vehicle_year' has accuracy: 0.6867
The column 'married' has accuracy: 0.6867
The column 'children' has accuracy: 0.6867
The column 'postal_code' has accuracy: 0.6867
The column 'annual_mileage' has accuracy: 0.6904
The column 'vehicle_type' has accuracy: 0.6867
The column 'speeding_violations' has accuracy: 0.6867
The column 'duis' has accuracy: 0.6867
The column 'past_accidents' has accuracy: 0.6867


In [6]:
## Finding the best performing model

# create variables for best model (the highest accuracy score)
best_model = features_cols[accuracy.index(max(accuracy))] 
# create DataFrame
best_feature_df = pd.DataFrame({'best_feature':best_model,'best_accuracy':max(accuracy)},index=[0])

print(f"The best feature to predict whether a customer will make a claim on their insurance during the policy period is '{best_feature_df['best_feature'][0]}' with accuracy: {best_feature_df['best_accuracy'][0]}")

display(best_feature_df)

The best feature to predict whether a customer will make a claim on their insurance during the policy period is 'driving_experience' with accuracy: 0.7771


,best_feature,best_accuracy
0,driving_experience,0.7771
